# CARLA360: inspect → run → visualize
Run cells one at a time. First choose the carla360 Python kernel. GPU inference is optional.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "run_eval.py").exists(), "Open this notebook inside the project"
sys.path.insert(0, str(ROOT))
from datasets.carla360 import Carla360Dataset
IMAGE_DIR = ROOT / "data/carla360/images"
POSE_CSV = ROOT / "data/carla360/positions_angles_camera1.csv"


In [ ]:
ds = Carla360Dataset(IMAGE_DIR, POSE_CSV)
print("Frames:", len(ds))
print("First records:", ds.manifest()[:3])


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(min(3,len(ds)), 1, figsize=(12, 7), squeeze=False)
for i, ax in enumerate(axes[:,0]):
    item = ds[i]
    ax.imshow(item["rgb"]); ax.set_title(f"Frame {item['frame_id']}: pitch,yaw,roll = {item['pitch_yaw_roll_deg']}"); ax.axis("off")
plt.tight_layout()


## GPU inference
Set a new run directory each time. This starts real inference and requires installed upstream dependencies and a checkpoint.

In [ ]:
import subprocess
RUN_DIR = ROOT / "outputs/panovggt01"
CHECKPOINT = ROOT / "checkpoints/panovggt.pt"
# Uncomment when GPU environment and checkpoint are ready:
# subprocess.run([sys.executable, "run_eval.py", "--image-dir", str(IMAGE_DIR),
#     "--pose-csv", str(POSE_CSV), "--checkpoint", str(CHECKPOINT),
#     "--limit", "3", "--output", str(RUN_DIR)], cwd=ROOT, check=True)


In [ ]:
import numpy as np
if (RUN_DIR / "predictions.npz").exists():
    with np.load(RUN_DIR / "predictions.npz") as pred:
        print({k: pred[k].shape for k in pred.files})
        if "depth" in pred:
            plt.figure(figsize=(12,4)); plt.imshow(np.squeeze(pred["depth"][0]), cmap="turbo"); plt.colorbar(); plt.title("Predicted depth: units not assumed metric")
else:
    print("No predictions yet; run GPU inference first.")
